# Task 2 Optuna screening on Kaggle
Start with classifier, then run separate specialist searches. Fresh searches do not need baseline checkpoints. Attach an extracted SEARCH backup only when continuing a paused search. Enable Internet and GPU. Run one search process at a time. Download the backup before ending the session.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


In [ ]:
import shutil
mode = 'classifier'  # classifier, salt, blur, occlusion: ONE search per invocation
assert mode in ['classifier', 'salt', 'blur', 'occlusion']
output = Path('/kaggle/working') / f'task2-optuna-{mode}'
# Later session: attach the extracted search backup; set the folder containing study.json.
resume_search_folder = None
if resume_search_folder is not None and not output.exists():
    source = Path(resume_search_folder)
    assert (source / 'study.json').is_file(), 'Use the search backup, not baseline model folder'
    shutil.copytree(source, output)


In [ ]:
module = 'optimization.optimize_task2_classifier' if mode == 'classifier' else 'optimization.optimize_task2_specialists'
command = [sys.executable, '-m', module, '--device', 'cuda', '--trials', '8',
           '--screen-epochs', '5', '--max-hours', '2', '--output-dir', str(output)]
if mode != 'classifier': command += ['--condition', mode]
subprocess.run(command, cwd=repo, check=True)


In [ ]:
import json
from IPython.display import display, FileLink
if (output / 'leaderboard.json').exists():
    leaderboard = json.loads((output / 'leaderboard.json').read_text())
    for trial in leaderboard['ranked_trials']:
        print(trial['number'], round(trial['score'],6), trial['params'])
else:
    print('No complete trial yet; preserve backup and resume the same search')
backup = shutil.make_archive(str(output) + '_backup', 'zip', output.parent, output.name)
display(FileLink(backup))


Eight is the TOTAL trial target, not additional trials. The two-hour budget is soft; rerun the same command to continue. Keep mode, source, config and screening budget unchanged for resume. Save the complete ZIP: study.json and trial checkpoints are both required. Upload to Drive manually. Share each leaderboard.json with its mode named. Do not run all four modes automatically: eight specialist trials × five epochs per condition is substantial compute.

Final screening epoch is ranked, not each trial's training-loss-selected best.pt. Classifier minimizes1-macroF1; specialists minimize0.8*L1+0.2*(1-SSIM), independent of training alpha. Trial0 repeats baseline settings. Screening winners are provisional; review before full training. Continue selected classifier trials with training.train_classifier_search and matching trial YAML/resume last.pt; specialists with training.train_specialists and matching condition/YAML/resume. Remove the stop limit but retain original total epochs. Official test set is unused.
